# Calibration refinement and nested cross-target experiment

The user reported a Kaggle score of **0.85329** for our previous candidate. This notebook records the next experiments; all new scores shown below are local. The new Kaggle score is unknown.

Run from the project root with the `.venv` interpreter. The v1 split IDs and v2 cached folds must exist; run `experiments.py` and `improvements.py` first on a fresh checkout. This notebook includes all new calibration, stacking, and orchestration code and reuses the earlier project helpers. No cells upload files.

Selection and stability checks reuse the 7,201 development rows and exclude the previously evaluated holdout. Final CV includes selection data. All scores must be interpreted with those limits.

## Calibration hypotheses

POC_GGT values are integers, with minimum 3. We test a detection-floor hypothesis rather than assuming all readings of 3 are exact measurements. Floor rows are excluded from batch-offset estimation; for known batches, the floor is treated as a one-sided observation with latent reading at most 3.5. A Gaussian clinical log-Gamma prior is updated using that bound. This distributional and floor assumption needs further confirmation; development evidence supports it here.

Nonfloor predictions combine calibrated readings with clinical regression. The weight uses training-only residual variance, effective batch sample count, and an approximate rounding variance. A fitted within-batch slope and fixed-weight alternatives are tested as ablations. Missing or unknown-batch readings retain the regression fallback.

In [ ]:
"""Fold-safe batch calibration with slope, rounding, and floor-value experiments."""
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.special import log_ndtr

import experiments as base


@dataclass
class Calibration:
    slope: float
    offsets: pd.Series
    effective_counts: pd.Series
    variance: float


def fit_calibration(reference, config):
    good = reference.POC_GGT.gt(0) & reference.POC_Batch.notna() & reference.Gamma_GT.gt(0)
    if config.get('exclude_floor', False):
        good &= reference.POC_GGT.gt(3)
    ref = reference.loc[good]
    if len(ref) == 0:
        return Calibration(1., pd.Series(dtype=float), pd.Series(dtype=float), .01)
    work = pd.DataFrame({'batch': ref.POC_Batch,
                         'x': np.log(ref.POC_GGT), 'y': np.log(ref.Gamma_GT),
                         'w': np.where(ref.POC_GGT.le(5), config.get('low_weight', 1.), 1.)})
    work['wx'] = work.w*work.x; work['wy'] = work.w*work.y
    work['ww'] = work.w**2
    grouped = work.groupby('batch')
    sums = grouped[['w', 'wx', 'wy', 'ww']].sum()
    mx = work.batch.map(sums.wx/sums.w); my = work.batch.map(sums.wy/sums.w)
    denominator = np.sum(work.w*(work.x-mx)**2)
    slope = float(np.sum(work.w*(work.x-mx)*(work.y-my))/denominator) if config.get('fit_slope', False) and denominator > 1e-9 else 1.
    # A failed or unsupported slope estimate must not create extreme predictions.
    slope = float(np.clip(slope, .5, 1.5))
    offsets = (sums.wy-slope*sums.wx)/sums.w
    residual = work.y-slope*work.x-work.batch.map(offsets)
    degrees = max(1, len(work)-len(sums)-int(config.get('fit_slope', False)))
    variance = max(1e-5, float(np.sum(work.w*residual**2)/work.w.sum()*len(work)/degrees))
    effective_counts = sums.w**2/sums.ww
    return Calibration(slope, offsets, effective_counts, variance)


def calibrated_prediction(calibration, frame, fallback, config):
    fallback = np.clip(np.asarray(fallback, dtype=float), 1, 1000)
    offset = frame.POC_Batch.map(calibration.offsets).to_numpy()
    counts = frame.POC_Batch.map(calibration.effective_counts).to_numpy()
    reading = frame.POC_GGT.to_numpy()
    known = (reading > 0) & np.isfinite(offset)
    floor = known & (reading <= 3) & config.get('exclude_floor', False)
    exact = known & ~floor
    p = fallback.copy()
    raw = np.clip(np.exp(offset[exact]+calibration.slope*np.log(reading[exact])), 1, 1000)
    if config.get('adaptive', False):
        measurement = calibration.variance*(1+1/counts[exact]) + calibration.slope**2/(12*reading[exact]**2)
        prior_variance = config.get('prior_sd', .27)**2
        weight = prior_variance/(prior_variance+measurement)
    else:
        weight = config.get('weight', .9)
    p[exact] = np.expm1(weight*np.log1p(raw)+(1-weight)*np.log1p(fallback[exact]))
    if config.get('floor_mode') == 'censored' and floor.any():
        # Hypothesis: a rounded reading of 3 represents a latent reading <=3.5.
        # Update the clinical log-Gamma prior using that one-sided observation.
        mu = np.log(fallback[floor])
        prior_variance = config.get('prior_sd', .27)**2
        measurement = calibration.variance*(1+1/counts[floor])
        sd = np.sqrt(prior_variance+measurement)
        upper = offset[floor]+calibration.slope*np.log(3.5)
        z = (upper-mu)/sd
        inverse_mills = np.exp(-.5*z*z-.5*np.log(2*np.pi)-log_ndtr(z))
        p[floor] = np.exp(mu-prior_variance/sd*inverse_mills)
    return np.clip(p, 1, 1000), known, floor


def candidate_configs():
    configs = {'v2': dict(fit_slope=False, exclude_floor=False, low_weight=1., weight=.9)}
    configs['slope'] = dict(fit_slope=True, exclude_floor=False, low_weight=1., weight=.9)
    for slope in [False, True]:
        for mode in ['fallback', 'censored']:
            for adaptive in [False, True]:
                name = f"{'slope' if slope else 'unit'}_floor_{mode}_{'adaptive' if adaptive else 'fixed'}"
                configs[name] = dict(fit_slope=slope, exclude_floor=True, low_weight=.3,
                                     weight=.9, floor_mode=mode, adaptive=adaptive, prior_sd=.27)
    return configs


def compare_cached(dev, cache, seed, gamma_name='gamma_clinical4'):
    from sklearn.model_selection import StratifiedKFold
    from improvements import ensemble, rmsle
    configs = candidate_configs()
    predictions = {name: np.zeros(len(dev)) for name in configs}
    smoke = np.zeros(len(dev)); fallback = np.zeros(len(dev)); floor = np.zeros(len(dev), dtype=bool)
    details = []
    for fold, (fit, valid) in enumerate(StratifiedKFold(3, shuffle=True, random_state=seed).split(dev, dev.Smoking)):
        ref, frame = dev.iloc[fit], dev.iloc[valid]
        smoking = {}
        for name in ['smoke_baseline', 'smoke_plate4', 'smoke_plate6']:
            saved = pd.read_csv(cache/f'{name}_fold{fold}.csv')
            assert saved.ID.tolist() == frame.ID.tolist()
            smoking[name] = saved.prediction.to_numpy()
        smoke[valid] = ensemble(smoking, list(smoking))
        saved = pd.read_csv(cache/f'{gamma_name}_fold{fold}.csv')
        assert saved.ID.tolist() == frame.ID.tolist()
        fallback[valid] = saved.prediction.to_numpy()
        floor[valid] = frame.POC_GGT.eq(3)
        for name, config in configs.items():
            calibration = fit_calibration(ref, config)
            predictions[name][valid], _, _ = calibrated_prediction(calibration, frame, fallback[valid], config)
            details.append(dict(fold=fold+1, name=name, slope=calibration.slope, variance=calibration.variance,
                                **base.metrics(frame, smoke[valid], predictions[name][valid])))
    rows = [dict(name=name, **config, **base.metrics(dev, smoke, predictions[name]),
                 floor_rmsle=rmsle(dev.Gamma_GT[floor], predictions[name][floor])) for name, config in configs.items()]
    return pd.DataFrame(rows), pd.DataFrame(details), predictions, smoke


## Nested cross-target features

Smoking probability and its log odds are candidate Gamma_GT features. Training probabilities are generated with three-fold cross-fitting. Each classifier's preprocessing and training exclude the predicted row. The Gamma_GT early-stopping pilot repeats this cross-fitting inside its own fitting partition, excluding stopping rows. Final refitting generates new OOF probabilities on the complete outer fitting partition. Outer evaluation uses a classifier trained only on that partition.

The crossfit audits store fitting and predicted IDs, allowing the exclusion requirement to be checked.

In [ ]:
"""Nested, label-isolated Smoking predictions for Gamma_GT regression."""
import json
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import StratifiedKFold, train_test_split

from improvements import engineered_features, fit_one, predict_one


def smoking_features(reference, directory, seed, full_model_path=None):
    """Every training probability excludes that row from model and preprocessing."""
    directory = Path(directory); directory.mkdir(parents=True, exist_ok=True)
    oof = np.zeros(len(reference)); audit = []
    for fold, (fit, valid) in enumerate(StratifiedKFold(3, shuffle=True, random_state=seed).split(reference, reference.Smoking)):
        ref, frame = reference.iloc[fit], reference.iloc[valid]
        assert set(ref.ID).isdisjoint(frame.ID)
        cache = directory/f'smoking_oof{fold}.csv'
        if cache.exists():
            saved = pd.read_csv(cache)
            assert saved.ID.tolist() == frame.ID.tolist()
            p = saved.prediction.to_numpy()
        else:
            model, trees = fit_one(ref, 'smoke_plate4', seed+fold)
            p = predict_one(model, ref, frame, 'smoke_plate4')
            pd.DataFrame({'ID':frame.ID, 'prediction':p}).to_csv(cache, index=False)
            model.save_model(str(directory/f'smoking_oof{fold}.cbm'))
        oof[valid] = p
        audit.append(dict(fold=fold, fitting_ids=ref.ID.tolist(), predicted_ids=frame.ID.tolist(), own_label_used=False))
    (directory/'crossfit_audit.json').write_text(json.dumps(audit))
    full_path = Path(full_model_path) if full_model_path else directory/'smoking_full.cbm'
    if full_path.exists():
        full = CatBoostClassifier(); full.load_model(str(full_path))
    else:
        full, trees = fit_one(reference, 'smoke_plate4', seed+20)
        full.save_model(str(full_path))
    assert np.isfinite(oof).all() and np.logical_and(oof>=0, oof<=1).all()
    return oof, full


def gamma_features(reference, frame, smoking):
    x, cats = engineered_features(reference, frame, 'clinical')
    p = np.clip(np.asarray(smoking), 1e-5, 1-1e-5)
    assert len(p) == len(frame)
    x['predicted_smoking'] = p
    x['predicted_smoking_log_odds'] = np.log(p/(1-p))
    return x, cats


def fit_stacked(reference, directory, seed, full_model_path=None):
    """Both inner early stopping and outer evaluation get isolated probabilities."""
    directory = Path(directory); directory.mkdir(parents=True, exist_ok=True)
    inner_fit, inner_stop = train_test_split(reference, test_size=.2,
                                           stratify=reference.Smoking, random_state=seed)
    # Do not generate early-stopping training features using stopping-row labels.
    inner_oof, inner_classifier = smoking_features(inner_fit, directory/'pilot_smoking', seed+50)
    stop_smoke = predict_one(inner_classifier, inner_fit, inner_stop, 'smoke_plate4')
    xi, cats = gamma_features(inner_fit, inner_fit, inner_oof)
    xs, _ = gamma_features(inner_fit, inner_stop, stop_smoke)
    params = dict(depth=4, learning_rate=.04, loss_function='RMSE', random_seed=seed,
                  cat_features=cats, verbose=False, allow_writing_files=False, thread_count=4)
    pilot = CatBoostRegressor(iterations=2400, **params)
    pilot.fit(xi, np.log1p(inner_fit.Gamma_GT), eval_set=(xs, np.log1p(inner_stop.Gamma_GT)), early_stopping_rounds=120)
    trees = max(1, pilot.get_best_iteration()+1)
    oof, classifier = smoking_features(reference, directory/'refit_smoking', seed+100, full_model_path)
    xf, _ = gamma_features(reference, reference, oof)
    model = CatBoostRegressor(iterations=trees, **params)
    model.fit(xf, np.log1p(reference.Gamma_GT))
    model.save_model(str(directory/'gamma_stacked.cbm'))
    (directory/'fit.json').write_text(json.dumps({'iterations':trees, 'seed':seed,
        'inner_fit_ids':inner_fit.ID.tolist(), 'inner_stop_ids':inner_stop.ID.tolist(),
        'training_smoking_predictions':'3-fold OOF', 'evaluation_smoking_predictions':'full-fitting-partition model'}))
    return model, classifier


def predict_stacked(model, classifier, reference, frame):
    p = predict_one(classifier, reference, frame, 'smoke_plate4')
    x, _ = gamma_features(reference, frame, p)
    return np.clip(np.expm1(model.predict(x)), 1, 1000)


## Select, check stability, and build predictions

The best calibration is frozen before a second development fold assignment. Promotion requires a score gain above 0.001 and gains in at least two folds. Stacking or a 50/50 regression blend must improve by more than 0.0005 before an additional stability check. Candidate selection does not use the user-reported public score.

In [ ]:
"""Refine the v2 candidate using calibration and nested cross-target features."""
import argparse
import hashlib
import importlib.metadata
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

import experiments as base
import improvements as previous
from advanced_calibration import fit_calibration, calibrated_prediction, compare_cached, candidate_configs
from cross_target import fit_stacked, predict_stacked

OUT = Path('outputs/plan_v3')
SEED = previous.SEED


def calibrate_experiment():
    dev = previous.development_data()
    rows, details, predictions, smoke = compare_cached(dev, previous.OUT/'selection', SEED)
    rows.to_csv(OUT/'calibration_selection.csv', index=False)
    details.to_csv(OUT/'calibration_selection_folds.csv', index=False)
    name = rows.sort_values('score', ascending=False).iloc[0]['name']
    config = dict(name=name, config=candidate_configs()[name], selected_before_confirmation=True)
    (OUT/'calibration_frozen.json').write_text(json.dumps(config, indent=2))
    rows, details, predictions, smoke = compare_cached(dev, previous.OUT/'confirmation', SEED+701)
    rows.to_csv(OUT/'calibration_confirmation.csv', index=False)
    details.to_csv(OUT/'calibration_confirmation_folds.csv', index=False)
    a = rows.set_index('name').loc[name]; b = rows.set_index('name').loc['v2']
    wins = sum(details[details.name.eq(name)].sort_values('fold').score.to_numpy() > details[details.name.eq('v2')].sort_values('fold').score.to_numpy())
    result = dict(selected_name=name, baseline=b[['auc','rmsle','score','floor_rmsle']].to_dict(),
                  selected=a[['auc','rmsle','score','floor_rmsle']].to_dict(), fold_wins=int(wins),
                  promotion_gate_passed=bool(a.score>b.score+.001 and wins>=2),
                  note='Second fold assignment on reused development rows; not independent validation.')
    (OUT/'calibration_confirmation.json').write_text(json.dumps(result, indent=2))
    print('CALIBRATION', json.dumps(result), flush=True)


def stage_predictions(dev, stage, seed):
    cache = previous.OUT/stage
    directory = OUT/stage; directory.mkdir(exist_ok=True)
    smoking = np.zeros(len(dev)); clinical = np.zeros(len(dev)); stacked = np.zeros(len(dev))
    assignments = np.zeros(len(dev), dtype=int)
    for fold, (fit, valid) in enumerate(StratifiedKFold(3, shuffle=True, random_state=seed).split(dev, dev.Smoking)):
        ref, frame = dev.iloc[fit], dev.iloc[valid]
        smoke_parts = {}
        for name in ['smoke_baseline','smoke_plate4','smoke_plate6']:
            saved = pd.read_csv(cache/f'{name}_fold{fold}.csv')
            assert saved.ID.tolist() == frame.ID.tolist()
            smoke_parts[name] = saved.prediction.to_numpy()
        smoking[valid] = previous.ensemble(smoke_parts, list(smoke_parts))
        saved = pd.read_csv(cache/f'gamma_clinical4_fold{fold}.csv')
        assert saved.ID.tolist() == frame.ID.tolist()
        clinical[valid] = saved.prediction.to_numpy()
        saved_path = directory/f'stack_fold{fold}.csv'
        if saved_path.exists():
            saved = pd.read_csv(saved_path)
            assert saved.ID.tolist() == frame.ID.tolist()
            prediction = saved.prediction.to_numpy()
        else:
            model, classifier = fit_stacked(ref, directory/f'stack_fold{fold}', seed+fold,
                                            cache/f'smoke_plate4_fold{fold}.cbm')
            prediction = predict_stacked(model, classifier, ref, frame)
            pd.DataFrame({'ID':frame.ID,'prediction':prediction}).to_csv(saved_path,index=False)
        stacked[valid] = prediction
        assignments[valid] = fold
        print(stage, 'stack fold', fold+1, 'regression RMSLE', previous.rmsle(frame.Gamma_GT, prediction), flush=True)
    return smoking, clinical, stacked, assignments


def regression_choices(clinical, stacked):
    return {'clinical':clinical, 'stacked':stacked,
            'blend':np.expm1(.5*np.log1p(clinical)+.5*np.log1p(stacked))}


def evaluate_choices(dev, smoking, fallbacks, assignments, seed, config):
    predictions = {name:np.zeros(len(dev)) for name in fallbacks}
    rows = []; folds = []; floor = dev.POC_GGT.eq(3).to_numpy()
    for fold, (fit, valid) in enumerate(StratifiedKFold(3, shuffle=True, random_state=seed).split(dev, dev.Smoking)):
        ref, frame = dev.iloc[fit], dev.iloc[valid]
        calibration = fit_calibration(ref, config)
        for name, fallback in fallbacks.items():
            predictions[name][valid], _, _ = calibrated_prediction(calibration, frame, fallback[valid], config)
            folds.append(dict(name=name,fold=fold+1,**base.metrics(frame,smoking[valid],predictions[name][valid])))
    missing = dev.POC_GGT.isna().to_numpy()
    for name, p in predictions.items():
        rows.append(dict(name=name,**base.metrics(dev, smoking, p),
                         regression_rmsle=previous.rmsle(dev.Gamma_GT, fallbacks[name]),
                         missing_reading_rmsle=previous.rmsle(dev.Gamma_GT[missing],p[missing]),
                         floor_rmsle=previous.rmsle(dev.Gamma_GT[floor],p[floor])))
    return pd.DataFrame(rows), pd.DataFrame(folds), predictions


def select():
    calibration = json.loads((OUT/'calibration_frozen.json').read_text())
    confirmed = json.loads((OUT/'calibration_confirmation.json').read_text())
    if not confirmed['promotion_gate_passed']:
        calibration = dict(name='v2',config=candidate_configs()['v2'])
    dev = previous.development_data()
    smoke, clinical, stacked, assignments = stage_predictions(dev, 'selection', SEED)
    rows, folds, predictions = evaluate_choices(dev, smoke, regression_choices(clinical,stacked),assignments,SEED,calibration['config'])
    rows.to_csv(OUT/'stack_selection.csv',index=False);folds.to_csv(OUT/'stack_selection_folds.csv',index=False)
    best = rows.sort_values('score',ascending=False).iloc[0]
    baseline = rows.set_index('name').loc['clinical']
    name = best['name'] if best.score > baseline.score+.0005 else 'clinical'
    config = dict(calibration=calibration,regression=name,stack_threshold=.0005,
                  selection=rows.to_dict(orient='records'),seed=SEED,
                  holdout_used_for_selection=False,
                  note='Calibration and stacking selected on previously used development rows.')
    (OUT/'frozen_config.json').write_text(json.dumps(config,indent=2))
    pd.DataFrame({'ID':dev.ID,'Smoking':smoke,'Gamma_GT':predictions[name]}).to_csv(OUT/'development_oof.csv',index=False)
    print('STACK SELECTION', rows.to_json(orient='records'), 'SELECTED',name,flush=True)


def confirm():
    config = json.loads((OUT/'frozen_config.json').read_text())
    if config['regression'] == 'clinical':
        (OUT/'stack_confirmation.json').write_text(json.dumps(dict(promoted=False,selected='clinical',
            note='No development gain exceeding 0.0005; omit stacking without additional tuning.'),indent=2))
        print('Stacking not selected; retain clinical regression.',flush=True)
        return
    dev = previous.development_data()
    smoke, clinical, stacked, assignments = stage_predictions(dev,'confirmation',SEED+701)
    rows, folds, predictions = evaluate_choices(dev,smoke,regression_choices(clinical,stacked),assignments,SEED+701,config['calibration']['config'])
    rows.to_csv(OUT/'stack_confirmation.csv',index=False);folds.to_csv(OUT/'stack_confirmation_folds.csv',index=False)
    name = config['regression'];index=rows.set_index('name')
    wins = sum(folds[folds.name.eq(name)].sort_values('fold').score.to_numpy()>folds[folds.name.eq('clinical')].sort_values('fold').score.to_numpy())
    passed = bool(index.loc[name,'score']>index.loc['clinical','score']+.0005 and wins>=2)
    result=dict(promoted=passed,selected=name if passed else 'clinical',fold_wins=int(wins),
                comparisons=rows.to_dict(orient='records'),
                note='Second development fold assignment; stability check, not an independent holdout.')
    (OUT/'stack_confirmation.json').write_text(json.dumps(result,indent=2))
    print('STACK CONFIRMATION',json.dumps(result),flush=True)


def finalize():
    config = json.loads((OUT/'frozen_config.json').read_text())
    stack_confirmation = json.loads((OUT/'stack_confirmation.json').read_text())
    regression = stack_confirmation['selected']
    train = pd.read_csv('data/train.csv');test=pd.read_csv('data/test.csv');sample=pd.read_csv('data/sample_submission.csv')
    original=pd.read_csv(previous.OUT/'final_oof.csv');old_submission=pd.read_csv(previous.OUT/'submission.csv')
    assert original.ID.equals(train.ID) and old_submission.ID.equals(test.ID)
    oof=np.zeros(len(train));test_log=np.zeros(len(test));covered=np.zeros(len(train),dtype=bool);floor=np.zeros(len(train),dtype=bool)
    directory=OUT/'final';directory.mkdir(exist_ok=True)
    for fold,(fit,valid) in enumerate(StratifiedKFold(5,shuffle=True,random_state=SEED).split(train,train.Smoking)):
        ref,frame=train.iloc[fit],train.iloc[valid]
        cached=np.load(previous.OUT/'final'/f'gamma_clinical4_fold{fold}.npz')
        assert np.array_equal(cached['valid_ids'],frame.ID) and np.array_equal(cached['test_ids'],test.ID)
        vg,tg=cached['valid'],cached['test']
        if regression != 'clinical':
            path=directory/f'stack_fold{fold}.npz'
            if path.exists():
                saved=np.load(path)
                assert np.array_equal(saved['valid_ids'],frame.ID) and np.array_equal(saved['test_ids'],test.ID)
                vs,ts=saved['valid'],saved['test']
            else:
                model,classifier=fit_stacked(ref,directory/f'stack_fold{fold}',SEED+200+fold,
                    previous.OUT/'final'/f'smoke_plate4_fold{fold}.cbm')
                vs=predict_stacked(model,classifier,ref,frame);ts=predict_stacked(model,classifier,ref,test)
                np.savez(path,valid=vs,test=ts,valid_ids=frame.ID,test_ids=test.ID)
            if regression == 'stacked': vg,tg=vs,ts
            else:
                vg=np.expm1(.5*np.log1p(vg)+.5*np.log1p(vs));tg=np.expm1(.5*np.log1p(tg)+.5*np.log1p(ts))
        calibration=fit_calibration(ref,config['calibration']['config'])
        vg,covered[valid],floor[valid]=calibrated_prediction(calibration,frame,vg,config['calibration']['config'])
        tg,_,_=calibrated_prediction(calibration,test,tg,config['calibration']['config'])
        oof[valid]=vg;test_log+=np.log1p(tg)/5
        (directory/f'calibration_fold{fold}.json').write_text(json.dumps(dict(slope=calibration.slope,
            variance=calibration.variance,offsets={str(k):float(v) for k,v in calibration.offsets.items()},
            effective_counts={str(k):float(v) for k,v in calibration.effective_counts.items()}),indent=2))
        print('Final fold',fold+1,base.metrics(frame,original.Smoking.iloc[valid],vg),flush=True)
    submission=pd.DataFrame({'ID':test.ID,'Smoking':old_submission.Smoking,'Gamma_GT':np.clip(np.expm1(test_log),1,1000)})
    assert submission.ID.equals(sample.ID) and len(submission)==6399
    assert submission.columns.tolist()==sample.columns.tolist()
    assert np.isfinite(submission[['Smoking','Gamma_GT']].to_numpy()).all()
    assert submission.Smoking.between(0,1).all() and submission.Gamma_GT.between(1,1000).all()
    submission.to_csv(OUT/'submission_v3.csv',index=False)
    pd.DataFrame({'ID':train.ID,'Smoking':original.Smoking,'Gamma_GT':oof,'calibrated':covered,'floor':floor}).to_csv(OUT/'final_oof.csv',index=False)
    slices=[]
    for name,mask in [('Nonfloor calibrated',covered & ~floor),('Floor=3',train.POC_GGT.eq(3).to_numpy()),
                      ('Fallback',~covered),('Gamma_GT >100',train.Gamma_GT.gt(100).to_numpy())]:
        slices.append(dict(subset=name,rows=int(mask.sum()),baseline_rmsle=previous.rmsle(train.Gamma_GT[mask],original.Gamma_GT[mask]),
                           improved_rmsle=previous.rmsle(train.Gamma_GT[mask],oof[mask])))
    pd.DataFrame(slices).to_csv(OUT/'final_error_slices.csv',index=False)
    report=dict(baseline=base.metrics(train,original.Smoking,original.Gamma_GT),selected=base.metrics(train,original.Smoking,oof),
                regression=regression,calibration=config['calibration'],
                user_reported_previous_kaggle_score=.85329,
                note='Final CV includes model-selection rows. New public score is unknown.')
    (OUT/'final_cv.json').write_text(json.dumps(report,indent=2))
    print('FINAL',json.dumps(report),flush=True)


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--phase',choices=['calibration','select','confirm','finalize','all'],default='all')
    args=parser.parse_args();OUT.mkdir(parents=True,exist_ok=True)
    identity=dict(code={p:hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in
        ['refine.py','advanced_calibration.py','cross_target.py','improvements.py','experiments.py']},
        data={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in Path('data').glob('*.csv')},
        versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','catboost']})
    manifest=OUT/'run_identity.json'
    if manifest.exists(): assert json.loads(manifest.read_text())==identity,'Recipe changed: use a new output directory.'
    else: manifest.write_text(json.dumps(identity,indent=2))
    if args.phase in ['calibration','all']:calibrate_experiment()
    if args.phase in ['select','all']:select()
    if args.phase in ['confirm','all']:confirm()
    if args.phase in ['finalize','all']:finalize()



In [ ]:
RUN_RETRAIN = False
if RUN_RETRAIN:
    import sys
    previous_argv = sys.argv
    try:
        sys.argv = ['refine.py', '--phase', 'all']
        main()
    finally:
        sys.argv = previous_argv
else:
    print('Displaying saved results. Set RUN_RETRAIN=True to reproduce the experiments.')

In [ ]:
from IPython.display import display
for filename in ['calibration_selection.csv', 'calibration_confirmation.csv', 'stack_selection.csv', 'final_error_slices.csv']:
    path = OUT / filename
    if path.exists():
        table = pd.read_csv(path)
        print(filename)
        display(table.sort_values('score', ascending=False) if 'score' in table else table)
    else:
        print(f'{path} is not available yet.')

In [ ]:
for filename in ['leaderboard_history.json', 'calibration_confirmation.json', 'stack_confirmation.json', 'paired_uncertainty.json', 'final_cv.json']:
    path = OUT / filename
    if path.exists():
        print(filename)
        display(json.loads(path.read_text()))

## Compare local regression errors

The final CV includes selection rows. The chart compares the previous candidate and updated calibration on exactly the same folds and classifier predictions.

In [ ]:
import matplotlib.pyplot as plt
path = OUT / 'final_error_slices.csv'
if path.exists():
    results = pd.read_csv(path)
    fig, ax = plt.subplots(figsize=(9, 4), layout='constrained')
    for i, row in results.iterrows():
        ax.plot([row.baseline_rmsle, row.improved_rmsle], [i, i], color='#cbd5e1', linewidth=3)
        ax.scatter(row.baseline_rmsle, i, color='#64748b', label='v2' if i == 0 else None)
        ax.scatter(row.improved_rmsle, i, color='#2563eb', label='v3' if i == 0 else None)
    ax.set_yticks(range(len(results)), [f'{r.subset} (n={r.rows:,})' for _, r in results.iterrows()])
    ax.invert_yaxis()
    ax.set_xlabel('Gamma_GT RMSLE — lower is better')
    ax.set_title('Final diagnostic CV: error by subset')
    ax.grid(axis='x', alpha=.2)
    ax.legend()
    fig.savefig(OUT / 'error_comparison.png', dpi=180)
    plt.show()

## Validate the next submission

The Smoking predictions are retained from v2. Only Gamma_GT changes. The file has a distinct name to avoid accidentally uploading the earlier candidate.

In [ ]:
path = OUT / 'submission_v3.csv'
if path.exists():
    submission = pd.read_csv(path)
    sample = pd.read_csv('data/sample_submission.csv')
    previous_submission = pd.read_csv('outputs/plan_v2/submission.csv')
    assert submission.ID.equals(sample.ID) and len(submission) == 6399
    assert submission.columns.tolist() == sample.columns.tolist()
    assert np.isfinite(submission[['Smoking', 'Gamma_GT']]).all().all()
    assert submission.Smoking.between(0, 1).all()
    assert submission.Gamma_GT.between(1, 1000).all()
    np.testing.assert_allclose(submission.Smoking, previous_submission.Smoking, rtol=1e-14, atol=1e-14)
    print(f'Validated {len(submission):,} predictions: {path}')
else:
    print('Run the workflow before uploading a new file.')